# Calculate country level mortality with parametric bootstrapping

This may take a while as it loops through each country 7x7x4 times (health_var x scenario x year)

In [1]:
import os
import numpy as np
import xarray as xr
from utils.mortality_utils import mortality
import config
from utils.utils import require_dir
import pathlib

In [2]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [3]:
# === Path config ===
POP_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "population")
BMR_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "BMR")
RR_DIR = require_dir(pathlib.Path(config.SCRATCH_ROOT) / "rr_pm25")

In [4]:
# === Calculate the scalar distributions ===
n_samples = 1000

In [5]:
# === Path config ===
MASK_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "BMR" / "masks" / "country")

in_file = "GBD_Country_Masks_EU_SHERPA_res.nc"
in_path = os.path.join(MASK_DIR, in_file)
country_mask = xr.open_dataarray(in_path)

In [ ]:
# === Scenario and path config ===
# For RR curves and file name
GBD_version = "GBD23"

scenarios = ["H", "HL", "L", "LN", "M", "ML", "VL"]
years = [2040, 2060, 2080, 2100]

PM25_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "SHERPA" / "processed")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "mortality" / "country" / f"{n_samples}_samples")

for health_VAR in health_vars:
    print(f"Processing health variable {health_VAR}")
    for scenario in scenarios:
        for year in years:
            print(f"Processing {scenario}, year {year}, {health_VAR}")

            # RR from GBD23 (normal distribution)
            rr_file = f"{GBD_version}_RR_{health_VAR}_{n_samples}_samples_pm25.nc"
            rr_path = os.path.join(RR_DIR, rr_file)
            rr_da = xr.open_dataarray(rr_path)

            del rr_file, rr_path

            # Load BMR for each grid point (normal distribution)
            bmr_file = f"{GBD_version}_BMR_Country_Mask_{health_VAR}_{n_samples}_samples_2015-2019.nc"
            bmr_path = os.path.join(BMR_DIR, bmr_file)
            BMR = xr.open_dataarray(bmr_path)

            del bmr_file, bmr_path

            pm25_file = f"EU_concentration_{scenario}_{year}.nc"
            pm25_path = os.path.join(PM25_DIR, pm25_file)
            pm25 = xr.open_dataarray(pm25_path)

            del pm25_file, pm25_path

            # Find the RR at each grid point
            RR = rr_da.interp(exposure=pm25)
            RR = RR.drop_vars(["exposure"])

            del pm25

            # Calculate the attributable fraction
            AF = (1 - (1/RR))

            del RR

            # Load population file
            pop_file = f"Population_count_regridded_{year}_{scenario}.nc"
            pop_path = os.path.join(POP_DIR, pop_file)
            POP = xr.open_dataarray(pop_path)

            # Calculate mortality at each grid point for n samples
            M = mortality(AF, BMR, POP)

            countries = []
            # Loop over countries and sum the mortality for each country
            for i in range(len(country_mask.country)):
                print(f"Country number {i}")
                mask = country_mask.isel(country=i)
                # Sum mortality of country (based on central estimates)
                M_country = (xr.where(
                    mask == 1,
                    M,
                    np.nan
                )).sum(dim=("latitude", "longitude"))
                countries.append(M_country.drop_vars("country", errors='ignore'))

            mortality_country = xr.concat(countries,
                                          dim=xr.DataArray(country_mask["country"],
                                                           dims="country",
                                                           name="country"))

            mortality_country.attrs["description"] = (f"Country level {health_VAR} Mortality due to "
                                                      "PM2.5 - scripts by A.F. Wells (2025)")
            mortality_country.attrs["scenario"] = scenario
            mortality_country.attrs["health_var"] = health_VAR
            mortality_country.attrs["GBD version"] = GBD_version

            out_file = f"Mortality_Country_sum_{GBD_version}_{health_VAR}_{scenario}_{year}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving country mortality to {out_path}")
            mortality_country.to_netcdf(out_path)

print("All processing complete.")

Processing health variable COPD
Processing H, year 2040, COPD
Country number 0
Country number 1
Country number 2
Country number 3
Country number 4
Country number 5
Country number 6
Country number 7
Country number 8
Country number 9
Country number 10
Country number 11
Country number 12
Country number 13
Country number 14
Country number 15
Country number 16
Country number 17
Country number 18
Country number 19
Country number 20
Country number 21
Country number 22
Country number 23
Country number 24
Country number 25
Country number 26
Country number 27
Country number 28
Country number 29
Country number 30
Country number 31
Country number 32
Country number 33
Country number 34
Country number 35
Country number 36
Country number 37
Country number 38
Country number 39
Country number 40
Country number 41
Country number 42
Country number 43
Country number 44
Country number 45
Country number 46
Country number 47
Country number 48
Country number 49
Country number 50
Country number 51
Country numb

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



Country number 159
Country number 160
Country number 161
Country number 162
Country number 163
Country number 164
Country number 165
Country number 166
Country number 167
Country number 168
Country number 169
Country number 170
Country number 171
Country number 172
Country number 173
Country number 174
Country number 175
Country number 176
Country number 177
Country number 178
Country number 179
Country number 180
Country number 181
Country number 182
Country number 183
Country number 184
Country number 185
Country number 186
Country number 187
Country number 188
Country number 189
Country number 190
Country number 191
Country number 192
Country number 193
Country number 194
Country number 195
Country number 196
Country number 197
Country number 198
Country number 199
Country number 200
Country number 201
Country number 202
Country number 203
Saving country mortality to /glade/work/awells/EU_pm/mortality/country/1000_samples/Mortality_Country_sum_GBD23_DIABETES_H_2080.nc
Processing H, 